Load transformed Silver data and create SQL view

In [0]:


# Define transformed Silver path
silver_path = (
    "/Volumes/workspace/default/railway_silver/transformed"
)

# Step 2: Read the transformed Delta dataset
silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

# Step 3: Create a temporary SQL view
# This allows Spark SQL queries against our DataFrame
silver_df.createOrReplaceTempView(
    "railway_silver"
)

# Step 4: Verify records and columns
print("Total Silver Records:", silver_df.count())
print("Total Columns:", len(silver_df.columns))

# Step 5: Test the SQL view
spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(DISTINCT train_no) AS total_trains,
        COUNT(DISTINCT station_code) AS total_stations,
        MIN(date) AS earliest_date,
        MAX(date) AS latest_date
    FROM railway_silver
""").show(truncate=False)

print("SQL VIEW CREATED SUCCESSFULLY")

Build overall railway performance KPIs

In [0]:
%sql


SELECT

    -- Total station-level observations
    COUNT(*) AS total_observations,

    -- Number of distinct trains
    COUNT(DISTINCT train_no) AS total_trains,

    -- Number of distinct railway stations
    COUNT(DISTINCT station_code) AS total_stations,

    -- Average verified arrival delay in minutes
    ROUND(
        AVG(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN calculated_arrival_delay_min
            END
        ), 2
    ) AS avg_arrival_delay_min,

    -- Average verified departure delay in minutes
    ROUND(
        AVG(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN calculated_departure_delay_min
            END
        ), 2
    ) AS avg_departure_delay_min,

    -- Arrival punctuality percentage
    -- EARLY and ON_TIME both count as punctual
    ROUND(
        100.0 * SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        ) /
        NULLIF(
            SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS arrival_punctuality_pct,

    -- Departure punctuality percentage
    ROUND(
        100.0 * SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        ) /
        NULLIF(
            SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS departure_punctuality_pct,

    -- Number of verified severely delayed arrivals
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
             AND arrival_delay_category = 'SEVERE_DELAY'
            THEN 1 ELSE 0
        END
    ) AS severe_arrival_count,

    -- Number of verified severely delayed departures
    SUM(
        CASE
            WHEN departure_delay_quality = 'MATCH'
             AND departure_delay_category = 'SEVERE_DELAY'
            THEN 1 ELSE 0
        END
    ) AS severe_departure_count,

    -- Records with inconsistent arrival delay values
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MISMATCH'
            THEN 1 ELSE 0
        END
    ) AS arrival_mismatch_count,

    -- Records with inconsistent departure delay values
    SUM(
        CASE
            WHEN departure_delay_quality = 'MISMATCH'
            THEN 1 ELSE 0
        END
    ) AS departure_mismatch_count

FROM railway_silver;

Station-wise delay performance analysis

In [0]:
%sql

SELECT

    -- Station identifier
    station_code,

    -- Total railway observations at the station
    COUNT(*) AS total_observations,

    -- Number of different trains observed
    COUNT(DISTINCT train_no) AS total_trains,

    -- Average verified arrival delay
    ROUND(
        AVG(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN calculated_arrival_delay_min
            END
        ), 2
    ) AS avg_arrival_delay_min,

    -- Average verified departure delay
    ROUND(
        AVG(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN calculated_departure_delay_min
            END
        ), 2
    ) AS avg_departure_delay_min,

    -- Arrival punctuality percentage
    ROUND(
        100.0 * SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        ) /
        NULLIF(
            SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS arrival_punctuality_pct,

    -- Count verified severe arrival delays
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
             AND arrival_delay_category = 'SEVERE_DELAY'
            THEN 1 ELSE 0
        END
    ) AS severe_arrival_count,

    -- Number of verified arrival observations
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
            THEN 1 ELSE 0
        END
    ) AS verified_arrival_count

FROM railway_silver

-- Group records by railway station
GROUP BY station_code

-- Avoid ranking stations with very few verified observations
HAVING SUM(
    CASE
        WHEN arrival_delay_quality = 'MATCH'
        THEN 1 ELSE 0
    END
) >= 30

-- Rank by highest average arrival delay
ORDER BY avg_arrival_delay_min DESC

-- Display the 10 most delay-prone stations
LIMIT 10;

Train-wise delay and punctuality analysis

In [0]:
%sql


SELECT

    -- Unique train identifier
    train_no,

    -- Train name
    MAX(train_name) AS train_name,

    -- Train category
    MAX(train_type) AS train_type,

    -- Total station observations
    COUNT(*) AS total_observations,

    -- Number of distinct stations covered
    COUNT(DISTINCT station_code) AS stations_covered,

    -- Average verified arrival delay
    ROUND(
        AVG(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN calculated_arrival_delay_min
            END
        ), 2
    ) AS avg_arrival_delay_min,

    -- Average verified departure delay
    ROUND(
        AVG(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN calculated_departure_delay_min
            END
        ), 2
    ) AS avg_departure_delay_min,

    -- Arrival punctuality percentage
    ROUND(
        100.0 * SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS arrival_punctuality_pct,

    -- Departure punctuality percentage
    ROUND(
        100.0 * SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS departure_punctuality_pct,

    -- Count severe arrival delays
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
             AND arrival_delay_category = 'SEVERE_DELAY'
            THEN 1 ELSE 0
        END
    ) AS severe_arrival_count,

    -- Count severe departure delays
    SUM(
        CASE
            WHEN departure_delay_quality = 'MATCH'
             AND departure_delay_category = 'SEVERE_DELAY'
            THEN 1 ELSE 0
        END
    ) AS severe_departure_count

FROM railway_silver

-- Group observations by train
GROUP BY train_no

-- Rank trains by highest average arrival delay
ORDER BY avg_arrival_delay_min DESC;

Monthly and Yearly Railway Delay Trends

In [0]:
%sql


SELECT

    -- Extract year from journey date
    YEAR(date) AS journey_year,

    -- Extract month from journey date
    MONTH(date) AS journey_month,

    -- Format month for dashboard display
    DATE_FORMAT(date, 'yyyy-MM') AS year_month,

    -- Total station observations in the month
    COUNT(*) AS total_observations,

    -- Number of distinct trains observed
    COUNT(DISTINCT train_no) AS total_trains,

    -- Average verified arrival delay
    ROUND(
        AVG(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN calculated_arrival_delay_min
            END
        ), 2
    ) AS avg_arrival_delay_min,

    -- Average verified departure delay
    ROUND(
        AVG(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN calculated_departure_delay_min
            END
        ), 2
    ) AS avg_departure_delay_min,

    -- Arrival punctuality percentage
    ROUND(
        100.0 * SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS arrival_punctuality_pct,

    -- Departure punctuality percentage
    ROUND(
        100.0 * SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS departure_punctuality_pct,

    -- Count verified severe arrival delays
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
             AND arrival_delay_category = 'SEVERE_DELAY'
            THEN 1 ELSE 0
        END
    ) AS severe_arrival_count,

    -- Count verified severe departure delays
    SUM(
        CASE
            WHEN departure_delay_quality = 'MATCH'
             AND departure_delay_category = 'SEVERE_DELAY'
            THEN 1 ELSE 0
        END
    ) AS severe_departure_count

FROM railway_silver

-- Group observations by year and month
GROUP BY
    YEAR(date),
    MONTH(date),
    DATE_FORMAT(date, 'yyyy-MM')

-- Display results chronologically
ORDER BY journey_year, journey_month;

Station-wise delay recovery analysis

In [0]:
%sql


SELECT

    -- Railway station identifier
    station_code,

    -- Total station observations
    COUNT(*) AS total_observations,

    -- Count records with both actual timestamps available
    SUM(
        CASE
            WHEN delay_recovery_status != 'UNKNOWN'
            THEN 1 ELSE 0
        END
    ) AS valid_recovery_observations,

    -- Average change in delay at the station
    -- Positive = delay reduced
    -- Negative = delay increased
    ROUND(
        AVG(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND departure_delay_quality = 'MATCH'
                THEN delay_recovery_min
            END
        ), 2
    ) AS avg_verified_recovery_min,

    -- Number of observations where delay decreased
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
             AND departure_delay_quality = 'MATCH'
             AND delay_recovery_status = 'RECOVERED'
            THEN 1 ELSE 0
        END
    ) AS recovered_count,

    -- Number of observations where delay increased
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
             AND departure_delay_quality = 'MATCH'
             AND delay_recovery_status = 'DELAY_INCREASED'
            THEN 1 ELSE 0
        END
    ) AS delay_increased_count,

    -- Number of observations where delay stayed unchanged
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
             AND departure_delay_quality = 'MATCH'
             AND delay_recovery_status = 'NO_CHANGE'
            THEN 1 ELSE 0
        END
    ) AS no_change_count,

    -- Percentage of verified observations with recovery
    ROUND(
        100.0 * SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND departure_delay_quality = 'MATCH'
                 AND delay_recovery_status = 'RECOVERED'
                THEN 1 ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                     AND departure_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS recovery_pct

FROM railway_silver

-- Aggregate station-level performance
GROUP BY station_code

-- Require at least 30 verified recovery observations
HAVING SUM(
    CASE
        WHEN arrival_delay_quality = 'MATCH'
         AND departure_delay_quality = 'MATCH'
        THEN 1 ELSE 0
    END
) >= 30

-- Rank stations by highest average verified recovery
ORDER BY avg_verified_recovery_min DESC

-- Display top 10 stations
LIMIT 10;

Year-wise railway performance

In [0]:
%sql


SELECT

    -- Extract the reporting year
    YEAR(date) AS journey_year,

    -- Total station-level observations
    COUNT(*) AS total_observations,

    -- Number of distinct trains
    COUNT(DISTINCT train_no) AS total_trains,

    -- Number of distinct stations
    COUNT(DISTINCT station_code) AS total_stations,

    -- Verified arrival observations
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MATCH'
            THEN 1 ELSE 0
        END
    ) AS verified_arrivals,

    -- Average verified arrival delay
    ROUND(
        AVG(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN calculated_arrival_delay_min
            END
        ), 2
    ) AS avg_arrival_delay_min,

    -- Average verified departure delay
    ROUND(
        AVG(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN calculated_departure_delay_min
            END
        ), 2
    ) AS avg_departure_delay_min,

    -- Percentage of punctual arrivals
    -- EARLY and ON_TIME count as punctual
    ROUND(
        100.0 * SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS arrival_punctuality_pct,

    -- Percentage of punctual departures
    ROUND(
        100.0 * SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category IN ('EARLY', 'ON_TIME')
                THEN 1 ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS departure_punctuality_pct,

    -- Percentage of verified arrivals delayed over 60 minutes
    ROUND(
        100.0 * SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN 1 ELSE 0
                END
            ), 0
        ), 2
    ) AS severe_arrival_pct,

    -- Count mismatched arrival delay observations
    SUM(
        CASE
            WHEN arrival_delay_quality = 'MISMATCH'
            THEN 1 ELSE 0
        END
    ) AS arrival_mismatch_count,

    -- Count mismatched departure delay observations
    SUM(
        CASE
            WHEN departure_delay_quality = 'MISMATCH'
            THEN 1 ELSE 0
        END
    ) AS departure_mismatch_count

FROM railway_silver

-- Group railway observations by year
GROUP BY YEAR(date)

-- Display results chronologically
ORDER BY journey_year;

first Gold Delta table

In [0]:
#  Gold Delta storage path
gold_kpi_path = (
    "/Volumes/workspace/default/"
    "railway_gold/overall_kpis"
)

# Calculate overall railway performance KPIs
gold_kpi_df = spark.sql("""
    SELECT

        -- Total observations
        COUNT(*) AS total_observations,

        -- Distinct trains and stations
        COUNT(DISTINCT train_no) AS total_trains,
        COUNT(DISTINCT station_code) AS total_stations,

        -- Average verified arrival delay
        ROUND(
            AVG(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN calculated_arrival_delay_min
                END
            ), 2
        ) AS avg_arrival_delay_min,

        -- Average verified departure delay
        ROUND(
            AVG(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN calculated_departure_delay_min
                END
            ), 2
        ) AS avg_departure_delay_min,

        -- Arrival punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                     AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN arrival_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS arrival_punctuality_pct,

        -- Departure punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                     AND departure_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN departure_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS departure_punctuality_pct,

        -- Severe arrival delays
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_arrival_count,

        -- Severe departure delays
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_departure_count,

        -- Delay quality mismatches
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MISMATCH'
                THEN 1 ELSE 0
            END
        ) AS arrival_mismatch_count,

        SUM(
            CASE
                WHEN departure_delay_quality = 'MISMATCH'
                THEN 1 ELSE 0
            END
        ) AS departure_mismatch_count

    FROM railway_silver
""")

# Save aggregated KPIs as a Delta dataset
# Overwrite supports repeatable full-refresh runs
gold_kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_kpi_path)

# Read Gold data back for verification
gold_kpi_check = spark.read \
    .format("delta") \
    .load(gold_kpi_path)

# Display saved KPI values
display(gold_kpi_check)

# Verify expected single-row aggregation
assert gold_kpi_check.count() == 1, (
    "Gold KPI table must contain exactly one row"
)

print("GOLD KPI DATASET CREATED SUCCESSFULLY")
print("Gold Path:", gold_kpi_path)
print("Total Gold Records:", gold_kpi_check.count())

Gold station-wise performance dataset

In [0]:
#  Gold storage location
gold_station_path = (
    "/Volumes/workspace/default/"
    "railway_gold/station_performance"
)

# Calculate station-wise performance metrics
gold_station_df = spark.sql("""
    SELECT

        -- Station identifier
        station_code,

        -- Total observations at each station
        COUNT(*) AS total_observations,

        -- Number of distinct trains
        COUNT(DISTINCT train_no) AS total_trains,

        -- Verified arrival observations
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN 1 ELSE 0
            END
        ) AS verified_arrival_count,

        -- Verified departure observations
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN 1 ELSE 0
            END
        ) AS verified_departure_count,

        -- Average verified arrival delay
        ROUND(
            AVG(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN calculated_arrival_delay_min
                END
            ), 2
        ) AS avg_arrival_delay_min,

        -- Average verified departure delay
        ROUND(
            AVG(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN calculated_departure_delay_min
                END
            ), 2
        ) AS avg_departure_delay_min,

        -- Arrival punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                     AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN arrival_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS arrival_punctuality_pct,

        -- Departure punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                     AND departure_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN departure_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS departure_punctuality_pct,

        -- Severe arrival delay count
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_arrival_count,

        -- Severe departure delay count
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_departure_count

    FROM railway_silver

    -- One row per station
    GROUP BY station_code
""")

# Save station performance as Delta
gold_station_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_station_path)

# Read saved Gold data
gold_station_check = spark.read \
    .format("delta") \
    .load(gold_station_path)

# Verify station count against Silver
expected_stations = silver_df.select(
    "station_code"
).distinct().count()

actual_stations = gold_station_check.count()

assert actual_stations == expected_stations, (
    f"Station count mismatch: "
    f"Expected {expected_stations}, got {actual_stations}"
)

# Display the 10 stations with highest average arrival delay
display(
    gold_station_check
    .filter("verified_arrival_count >= 30")
    .orderBy(
        "avg_arrival_delay_min",
        ascending=False
    )
    .limit(10)
)

print("GOLD STATION DATASET CREATED SUCCESSFULLY")
print("Gold Path:", gold_station_path)
print("Total Stations:", actual_stations)
print("Total Columns:", len(gold_station_check.columns))

Gold Train-wise Performance

In [0]:


# Define Gold Delta storage path
gold_train_path = (
    "/Volumes/workspace/default/"
    "railway_gold/train_performance"
)

# Calculate train-wise performance metrics
gold_train_df = spark.sql("""
    SELECT

        -- Unique train identifier
        train_no,

        -- Train information
        MAX(train_name) AS train_name,
        MAX(train_type) AS train_type,

        -- Total station-level observations
        COUNT(*) AS total_observations,

        -- Number of distinct stations covered
        COUNT(DISTINCT station_code) AS stations_covered,

        -- Verified arrival observations
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN 1 ELSE 0
            END
        ) AS verified_arrival_count,

        -- Verified departure observations
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN 1 ELSE 0
            END
        ) AS verified_departure_count,

        -- Average verified arrival delay
        ROUND(
            AVG(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN calculated_arrival_delay_min
            END
        ), 2
        ) AS avg_arrival_delay_min,

        -- Average verified departure delay
        ROUND(
            AVG(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN calculated_departure_delay_min
            END
        ), 2
        ) AS avg_departure_delay_min,

        -- Arrival punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                     AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN arrival_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS arrival_punctuality_pct,

        -- Departure punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                     AND departure_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN departure_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS departure_punctuality_pct,

        -- Severe arrival delay count
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_arrival_count,

        -- Severe departure delay count
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_departure_count

    FROM railway_silver

    -- One aggregated row per train
    GROUP BY train_no
""")

# Save Gold train performance as Delta
gold_train_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_train_path)

# Read the saved Gold dataset
gold_train_check = spark.read \
    .format("delta") \
    .load(gold_train_path)

# Calculate expected train count from Silver
expected_trains = silver_df.select(
    "train_no"
).distinct().count()

# Calculate actual Gold train count
actual_trains = gold_train_check.count()

# Validate the record count
assert actual_trains == expected_trains, (
    f"Train count mismatch: "
    f"Expected {expected_trains}, got {actual_trains}"
)

# Display trains ranked by average arrival delay
display(
    gold_train_check.orderBy(
        "avg_arrival_delay_min",
        ascending=False
    )
)

# Print validation results
print("GOLD TRAIN DATASET CREATED SUCCESSFULLY")
print("Gold Path:", gold_train_path)
print("Total Trains:", actual_trains)
print("Total Columns:", len(gold_train_check.columns))

Gold Monthly Performance

In [0]:


# Define Gold Delta storage path
gold_monthly_path = (
    "/Volumes/workspace/default/"
    "railway_gold/monthly_performance"
)

# Calculate monthly railway performance metrics
gold_monthly_df = spark.sql("""
    SELECT

        -- Reporting year and month
        YEAR(date) AS journey_year,
        MONTH(date) AS journey_month,

        -- First day of month for Power BI date filtering
        DATE_TRUNC('MONTH', date) AS month_start,

        -- Readable month label
        DATE_FORMAT(date, 'yyyy-MM') AS year_month,

        -- Total station-level observations
        COUNT(*) AS total_observations,

        -- Distinct trains and stations observed
        COUNT(DISTINCT train_no) AS total_trains,
        COUNT(DISTINCT station_code) AS total_stations,

        -- Number of verified arrival observations
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN 1 ELSE 0
            END
        ) AS verified_arrival_count,

        -- Number of verified departure observations
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN 1 ELSE 0
            END
        ) AS verified_departure_count,

        -- Average verified arrival delay
        ROUND(
            AVG(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN calculated_arrival_delay_min
                END
            ), 2
        ) AS avg_arrival_delay_min,

        -- Average verified departure delay
        ROUND(
            AVG(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN calculated_departure_delay_min
                END
            ), 2
        ) AS avg_departure_delay_min,

        -- Arrival punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                     AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN arrival_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS arrival_punctuality_pct,

        -- Departure punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                     AND departure_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN departure_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS departure_punctuality_pct,

        -- Count severe arrival delays
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_arrival_count,

        -- Count severe departure delays
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_departure_count

    FROM railway_silver

    -- Aggregate observations by calendar month
    GROUP BY
        YEAR(date),
        MONTH(date),
        DATE_TRUNC('MONTH', date),
        DATE_FORMAT(date, 'yyyy-MM')
""")

# Save the monthly aggregates as Delta
gold_monthly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_monthly_path)

# Read the saved Gold dataset
gold_monthly_check = spark.read \
    .format("delta") \
    .load(gold_monthly_path)

# Count expected distinct months from Silver
expected_months = silver_df.selectExpr(
    "DATE_TRUNC('MONTH', date) AS month_start"
).distinct().count()

# Count actual Gold monthly records
actual_months = gold_monthly_check.count()

# Validate monthly record count
assert actual_months == expected_months, (
    f"Monthly count mismatch: "
    f"Expected {expected_months}, got {actual_months}"
)

# Verify that all Silver observations are represented
gold_total = gold_monthly_check.agg(
    {"total_observations": "sum"}
).first()[0]

silver_total = silver_df.count()

assert gold_total == silver_total, (
    f"Observation mismatch: "
    f"Silver={silver_total}, Gold={gold_total}"
)

# Display monthly performance chronologically
display(
    gold_monthly_check.orderBy(
        "journey_year",
        "journey_month"
    )
)

# Print validation results
print("GOLD MONTHLY DATASET CREATED SUCCESSFULLY")
print("Gold Path:", gold_monthly_path)
print("Total Months:", actual_months)
print("Total Columns:", len(gold_monthly_check.columns))
print("Reconciliation:", gold_total, "=", silver_total)

Gold Yearly Performance

In [0]:


# Define Gold Delta storage path
gold_yearly_path = (
    "/Volumes/workspace/default/"
    "railway_gold/yearly_performance"
)

# Calculate yearly railway performance metrics
gold_yearly_df = spark.sql("""
    SELECT

        -- Reporting year
        YEAR(date) AS journey_year,

        -- Total station-level observations
        COUNT(*) AS total_observations,

        -- Number of distinct trains
        COUNT(DISTINCT train_no) AS total_trains,

        -- Number of distinct stations
        COUNT(DISTINCT station_code) AS total_stations,

        -- Verified arrival observations
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                THEN 1 ELSE 0
            END
        ) AS verified_arrival_count,

        -- Verified departure observations
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                THEN 1 ELSE 0
            END
        ) AS verified_departure_count,

        -- Average verified arrival delay
        ROUND(
            AVG(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                    THEN calculated_arrival_delay_min
                END
            ), 2
        ) AS avg_arrival_delay_min,

        -- Average verified departure delay
        ROUND(
            AVG(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                    THEN calculated_departure_delay_min
                END
            ), 2
        ) AS avg_departure_delay_min,

        -- Arrival punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                     AND arrival_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN arrival_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS arrival_punctuality_pct,

        -- Departure punctuality percentage
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN departure_delay_quality = 'MATCH'
                     AND departure_delay_category IN ('EARLY', 'ON_TIME')
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN departure_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS departure_punctuality_pct,

        -- Percentage of verified arrivals severely delayed
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN arrival_delay_quality = 'MATCH'
                     AND arrival_delay_category = 'SEVERE_DELAY'
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(
                    CASE
                        WHEN arrival_delay_quality = 'MATCH'
                        THEN 1 ELSE 0
                    END
                ), 0
            ), 2
        ) AS severe_arrival_pct,

        -- Severe arrival delay count
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND arrival_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_arrival_count,

        -- Severe departure delay count
        SUM(
            CASE
                WHEN departure_delay_quality = 'MATCH'
                 AND departure_delay_category = 'SEVERE_DELAY'
                THEN 1 ELSE 0
            END
        ) AS severe_departure_count,

        -- Arrival delay quality mismatches
        SUM(
            CASE
                WHEN arrival_delay_quality = 'MISMATCH'
                THEN 1 ELSE 0
            END
        ) AS arrival_mismatch_count,

        -- Departure delay quality mismatches
        SUM(
            CASE
                WHEN departure_delay_quality = 'MISMATCH'
                THEN 1 ELSE 0
            END
        ) AS departure_mismatch_count

    FROM railway_silver

    -- One aggregated row per reporting year
    GROUP BY YEAR(date)
""")

# Save yearly performance in Delta format
gold_yearly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_yearly_path)

# Read saved Gold dataset
gold_yearly_check = spark.read \
    .format("delta") \
    .load(gold_yearly_path)

# Calculate expected number of years from Silver
expected_years = silver_df.selectExpr(
    "YEAR(date) AS journey_year"
).distinct().count()

# Calculate actual Gold records
actual_years = gold_yearly_check.count()

# Validate yearly record count
assert actual_years == expected_years, (
    f"Year count mismatch: "
    f"Expected {expected_years}, got {actual_years}"
)

# Reconcile Gold observations with Silver
gold_total = gold_yearly_check.agg(
    {"total_observations": "sum"}
).first()[0]

silver_total = silver_df.count()

assert gold_total == silver_total, (
    f"Observation mismatch: "
    f"Silver={silver_total}, Gold={gold_total}"
)

# Display year-wise performance
display(
    gold_yearly_check.orderBy("journey_year")
)

# Print validation results
print("GOLD YEARLY DATASET CREATED SUCCESSFULLY")
print("Gold Path:", gold_yearly_path)
print("Total Years:", actual_years)
print("Total Columns:", len(gold_yearly_check.columns))
print("Reconciliation:", gold_total, "=", silver_total)

Gold Station Delay Recovery

In [0]:


#  Gold Delta storage path
gold_recovery_path = (
    "/Volumes/workspace/default/"
    "railway_gold/station_recovery"
)

# Calculate station-level delay recovery metrics
gold_recovery_df = spark.sql("""
    WITH recovery_base AS (

        SELECT

            -- Station identifier
            station_code,

            -- Delay recovery calculated in Silver
            delay_recovery_min,
            delay_recovery_status,

            -- Only accept observations where both
            -- arrival and departure delays are verified
            CASE
                WHEN arrival_delay_quality = 'MATCH'
                 AND departure_delay_quality = 'MATCH'
                 AND delay_recovery_min IS NOT NULL
                THEN 1
                ELSE 0
            END AS is_verified_recovery

        FROM railway_silver
    )

    SELECT

        -- Railway station identifier
        station_code,

        -- All observations at this station
        COUNT(*) AS total_observations,

        -- Number of verified recovery observations
        SUM(is_verified_recovery)
            AS verified_recovery_count,

        -- Average delay recovery in minutes
        -- Positive = delay reduced
        -- Negative = delay increased
        ROUND(
            AVG(
                CASE
                    WHEN is_verified_recovery = 1
                    THEN delay_recovery_min
                END
            ), 2
        ) AS avg_recovery_min,

        -- Number of observations where delay reduced
        SUM(
            CASE
                WHEN is_verified_recovery = 1
                 AND delay_recovery_status = 'RECOVERED'
                THEN 1 ELSE 0
            END
        ) AS recovered_count,

        -- Number of observations where delay increased
        SUM(
            CASE
                WHEN is_verified_recovery = 1
                 AND delay_recovery_status = 'DELAY_INCREASED'
                THEN 1 ELSE 0
            END
        ) AS delay_increased_count,

        -- Number of observations with unchanged delay
        SUM(
            CASE
                WHEN is_verified_recovery = 1
                 AND delay_recovery_status = 'NO_CHANGE'
                THEN 1 ELSE 0
            END
        ) AS no_change_count,

        -- Percentage of verified observations
        -- where trains recovered delay
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN is_verified_recovery = 1
                     AND delay_recovery_status = 'RECOVERED'
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(is_verified_recovery), 0
            ), 2
        ) AS recovery_pct,

        -- Percentage of verified observations
        -- where delay increased
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN is_verified_recovery = 1
                     AND delay_recovery_status = 'DELAY_INCREASED'
                    THEN 1 ELSE 0
                END
            )
            /
            NULLIF(
                SUM(is_verified_recovery), 0
            ), 2
        ) AS delay_increased_pct

    FROM recovery_base

    -- One row per station
    GROUP BY station_code
""")

# Save recovery metrics as a Delta dataset
gold_recovery_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_recovery_path)

# Read the saved Gold dataset
gold_recovery_check = spark.read \
    .format("delta") \
    .load(gold_recovery_path)

# Calculate expected station count from Silver
expected_stations = silver_df.select(
    "station_code"
).distinct().count()

# Count actual Gold station records
actual_stations = gold_recovery_check.count()

# Validate station count
assert actual_stations == expected_stations, (
    f"Station count mismatch: "
    f"Expected {expected_stations}, "
    f"Got {actual_stations}"
)

# Verify all Silver observations are represented
gold_total = gold_recovery_check.agg(
    {"total_observations": "sum"}
).first()[0]

silver_total = silver_df.count()

assert gold_total == silver_total, (
    f"Reconciliation failed: "
    f"Silver={silver_total}, Gold={gold_total}"
)

# Verify recovery status counts reconcile
invalid_recovery_groups = gold_recovery_check.filter("""
    recovered_count
    + delay_increased_count
    + no_change_count
    != verified_recovery_count
""").count()

assert invalid_recovery_groups == 0, (
    "Recovery status counts do not reconcile"
)

# Display top 10 stations by average recovery
# Minimum 30 verified observations for ranking
display(
    gold_recovery_check
    .filter("verified_recovery_count >= 30")
    .orderBy(
        "avg_recovery_min",
        ascending=False
    )
    .limit(10)
)

# Print validation results
print("GOLD RECOVERY DATASET CREATED SUCCESSFULLY")
print("Gold Path:", gold_recovery_path)
print("Total Stations:", actual_stations)
print("Total Columns:", len(gold_recovery_check.columns))
print("Reconciliation:", gold_total, "=", silver_total)
print("Recovery Status Validation: PASSED")